<a href="https://colab.research.google.com/github/ge56sob/PC-reasoning-llms-for-underrepresented-languages/blob/main/Cuong%20-%20Small%20-%20Model%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets huggingface_hub
!pip install -U transformers

In [10]:


from datasets import load_dataset, get_dataset_config_names
from google.colab import userdata
from huggingface_hub import login

# Login
hf_token = userdata.get("HuggingFace").strip()
login(token=hf_token)


# Check available configs
configs = get_dataset_config_names("Qwen/PolyMath")

# Load datasets
ds_vi = load_dataset("Qwen/PolyMath", "vi")
ds_en = load_dataset("Qwen/PolyMath", "en")

#print(ds_vi)
#print(ds_en)


In [ ]:
"""
for level in ds_vi.keys():

    print(f"\n========== {level.upper()} ==========\n")

    dataset = ds_vi[level]

    for i in range(len(dataset)):

        print(f"Row {i}")
        print("ID:", dataset[i]["id"])
        print("Question:", dataset[i]["question"])
        print("Answer:", dataset[i]["answer"])
        print("-" * 80)
"""

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="sail/Sailor2-1B-Chat")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("sail/Sailor2-1B-Chat")
model = AutoModelForCausalLM.from_pretrained("sail/Sailor2-1B-Chat")


In [ ]:
messages = [
    {"role": "user", "content": "Một nhân viên hải quan tại cảng chính của bãi biển đã kiểm đếm 2 container xe đạp nhập khẩu, mỗi container chứa 5 chiếc xe đạp. Ngày hôm sau, có thêm nhiều container được đưa đến, và tổng số xe đạp tại cảng đã lên tới 30 chiếc. Hãy tính số lượng container nhập khẩu vào ngày thứ hai, giả sử tất cả các container đều chứa 5 chiếc xe đạp. Lưu ý: Vui lòng đặt câu trảlời cuối cùng trong $\\boxed{}$."},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=4000)
#print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [15]:
messages = [
    {"role": "user", "content": "A customs officer at the main port of the beach counted 2 containers of imported bicycles, with each container containing 5 bicycles. The next day, more containers arrived, and the total number of bicycles at the port reached 30. Calculate the number of containers imported on the second day, assuming that all containers also contained 5 bicycles each. Note: Please put the final answer in $\\boxed{}$"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=4000)
#print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
for level in ds_vi.keys():

    print(f"\n========== {level.upper()} ==========\n")

    if level == "low":
      dataset = ds_vi[level]
      for i in range(len(dataset)):

          print("ID:", dataset[i]["id"])
          print("Question:", dataset[i]["question"])
          messages = [
            {"role": "user", "content": dataset[i]["question"] + "\n\nLưu ý: Vui lòng đặt câu trảlời cuối cùng trong $\\boxed{}$."}
          ]
          inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
          ).to(model.device)

          outputs = model.generate(**inputs, max_new_tokens=1000)
          print("Model's Answer:", tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
          print("Original Answer:", dataset[i]["answer"])
          print("-" * 80)


========== TOP ==========


========== HIGH ==========


========== MEDIUM ==========


========== LOW ==========

ID: low-vi-0
Question: Con vịt của Janet mỗi ngày đẻ 16 quả trứng. Cô ấy ăn ba quả vào bữa sáng mỗi ngày và làm bánh nướng cho bạn bè của cô ấy mỗi ngày với bốn quả trứng. Cô ấy bán số còn lại tại chợ nông sản hàng ngày với giá 2 đô la mỗi quả trứng vịt tươi. Cô ấy kiếm được bao nhiêu đô la mỗi ngày tại chợ nông sản?
Model's Answer: **Janet kiếm được $78$ đô la mỗi ngày tại chợ nông sản.

Dữ liệu từng bước:

1. **Đơn vị trứng vịt:** Mỗi con vịt đẻ 16 quả.
2. **Trứng ăn mỗi ngày:** 
   - Buổi sáng: 3 quả
   - Bữa trưa: 4 quả
3. **Bánh nướng:** 3 quả/ngày x 3 ngày = 9 quả (công việc)
4. **Số trứng còn lại:** \(16 \text{ quả} - 3 \text{ quả (bữa sáng)} - 9 \text{ quả (trưa)} = 5 \text{ quả}\)

5. **Giá trị bán:** 5 quả * $0.2/\text{đô la} = $1.00

6. **Tổng thu nhập từ thị trường**: Số trứng còn lại + Thu nhập từ bán trứng = $5 + $1 = $6

7. **Chi phí ăn uống**: 3 quả * $0.3